# Test the way the world files are read

In [2]:
import os

path = "/root/e2e_crossq/src/the-barn-challenge-CrossQ/jackal_helper/worlds/BARN"
os.listdir(path)

['world_218.world',
 'world_191.world',
 'world_263.world',
 'world_271.world',
 'world_100.world',
 'world_138.world',
 'world_35.world',
 'world_18.world',
 'world_80.world',
 'world_66.world',
 'world_91.world',
 'world_281.world',
 'world_26.world',
 'world_180.world',
 'world_275.world',
 'world_34.world',
 'world_81.world',
 'world_14.world',
 'world_221.world',
 'world_202.world',
 'world_88.world',
 'world_21.world',
 'world_107.world',
 'world_170.world',
 'world_8.world',
 'world_289.world',
 'world_254.world',
 'world_118.world',
 'world_208.world',
 'world_294.world',
 'world_220.world',
 'world_40.world',
 'world_286.world',
 'world_205.world',
 'world_7.world',
 'world_17.world',
 'world_282.world',
 'world_285.world',
 'world_250.world',
 'world_95.world',
 'world_144.world',
 'world_146.world',
 'world_216.world',
 'world_203.world',
 'world_152.world',
 'world_85.world',
 'world_128.world',
 'world_122.world',
 'world_84.world',
 'world_33.world',
 'world_210.world',
 

# Test the encoders and all ML part

In [1]:
import torch 
from sac.net import TCNEncoder

state = torch.ones([2, 4, 724])
net = TCNEncoder((4, 720), 2, 512)
no_laser_data = state[:, :, -4:].reshape(state.shape[0], -1)
state = state[:, :, :-4]
s = net(state)
print('State: ', s.shape)
print('No laser data: ', no_laser_data.shape)
s = torch.cat([s, no_laser_data], dim=1)
s.shape

State:  torch.Size([2, 720])
No laser data:  torch.Size([2, 16])


torch.Size([2, 736])

In [3]:
#from envs.multi_reward import MultiRewardEnv
import gym
from gym.spaces import Box
import numpy as np
import torch

#env = gym.make('MultiRewardEnv-v0')

min_v=-1
max_v=2
min_w=-3.14
max_w=3.14

range_dict = RANGE_DICT = {
            "linear_velocity": [min_v, max_v],
            "angular_velocity": [min_w, max_w],
        }

action_space = Box(
            low=np.array([RANGE_DICT["linear_velocity"][0], RANGE_DICT["angular_velocity"][0]]),
            high=np.array([RANGE_DICT["linear_velocity"][1], RANGE_DICT["angular_velocity"][1]]),
            dtype=np.float32
        )

actions = torch.Tensor([[action_space.sample()] for i in range(2)]).squeeze(1)
print(actions)
print(actions.shape)

final_input = torch.cat([s, actions], dim=1)

tensor([[-0.9654, -1.9815],
        [ 0.4628,  2.6997]])
torch.Size([2, 2])


In [11]:
from sac.net import TCNEncoder, MLP_CrossQ
from sac.rl import CrossQCritic, Actor
import torch
from torch.nn import functional as F

state = torch.ones([2, 4, 724])
action = torch.Tensor([[ 1.0215,  1.5293], [-0.8067,  0.3733]])

action_dim = np.prod(action_space.shape)
action_space_low = action_space.low
action_space_high = action_space.high

input_dim = 736 # Input dim is [laser dimensio + stack frames*size of local goal + stack frames*action dim]
actor = Actor(
    state_preprocess=TCNEncoder((4, 720), 2, 512),
    head=MLP_CrossQ(
        input_dim,
        2,
        512,
    ),
    action_dim=action_dim,
    action_space_high=action_space_high, 
    action_space_low=action_space_low
)

_, log_probs, _ = actor.get_action_alt(state)


critic = CrossQCritic(
        state_preprocess=TCNEncoder((4, 720), 2, 512),
        head=MLP_CrossQ(
            738,
            2,
            512,
        ))

cat_state = torch.cat([state, state], dim=0)
cat_actions = torch.cat([action, action], dim=0)
cat_q1, cat_q2 = critic(cat_state, cat_actions)

q_values_1, q_values_1_next = torch.chunk(cat_q1, chunks=2, dim=0)
q_values_2, q_values_2_next = torch.chunk(cat_q2, chunks=2, dim=0)

# print(q_values_1)
# print(q_values_1.detach().mean())
# print(q_values_2)

init_temperature = 1.0

log_alpha = torch.tensor(
    [np.log(init_temperature)],
    requires_grad=True,
    dtype=torch.float32,
)

target_q_values = (
    torch.minimum(q_values_1_next, q_values_2_next)
    - log_alpha.exp() * log_probs
)

print("q_values_1 shape:", q_values_1.shape)
print("q_values_2 shape:", q_values_2.shape)
print("q_values_1_next shape:", q_values_1_next.shape)
print("q_values_2_next shape:", q_values_2_next.shape)
print("target_q_values shape:", target_q_values.shape)

rewards_scale = 1.0
reward = torch.ones([2, 1])
termination = torch.zeros([2, 1])
gamma = 0.99

q_target = (
    reward * rewards_scale
    + gamma * (1 - termination) * target_q_values
).detach()

print("q_target shape:", q_target.shape)

q1_loss = F.mse_loss(q_values_1, q_target)
q2_loss = F.mse_loss(q_values_2, q_target)
total_q_loss = q1_loss + q2_loss

q_values_1 shape: torch.Size([2, 1])
q_values_2 shape: torch.Size([2, 1])
q_values_1_next shape: torch.Size([2, 1])
q_values_2_next shape: torch.Size([2, 1])
target_q_values shape: torch.Size([2, 1])
q_target shape: torch.Size([2, 1])
